# RAG with MongoDB Atlas — Improved Version
### Improvements inspired by DualLens Analytics architecture:
- ✅ Metadata tagging per chunk (source, page, topic)
- ✅ Filtered retrieval using MongoDB `pre_filter`
- ✅ LLM answer synthesis layer (not just raw chunks)
- ✅ LLM-as-Judge: Groundedness + Relevance evaluation
- ✅ Evaluation scores stored back to MongoDB
- ✅ Score visualisation dashboard
- ✅ Reusable, clean function structure


## Cell 1 — Installation
> Run once, then restart the session before continuing.

In [ ]:
# @title Run once => Restart session => Continue with Cell 2
!pip install pymongo langchain langchain-community langchain-mongodb \
            langchain-openai langchain-huggingface \
            sentence-transformers pypdf matplotlib pandas -q


## Cell 2 — Imports

In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

from pymongo import MongoClient
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

print("✓ All imports successful")


## Cell 3 — Configuration
Set your MongoDB URI and (optionally) your OpenAI key for the LLM answer + judge layer.
The embedding model remains **free** (HuggingFace).


In [ ]:
# ── Option A: load from config.json ──────────────────────────────────────
# Uncomment if you have a config.json with MONGODB_URI and OPENAI_API_KEY
# with open("config.json") as f:
#     cfg = json.load(f)
#     MONGODB_URI   = cfg["MONGODB_URI"]
#     OPENAI_API_KEY = cfg["OPENAI_API_KEY"]
#     os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ── Option B: paste directly (do NOT commit to GitHub) ───────────────────
MONGODB_URI    = "your-mongodb-atlas-uri-here"
OPENAI_API_KEY = "your-openai-api-key-here"   # needed for LLM + judge
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ── MongoDB settings ──────────────────────────────────────────────────────
DB_NAME         = "book_mongodb_chunks"
COLLECTION_NAME = "chunked_data"
INDEX_NAME      = "vector_index"
EVAL_COLLECTION = "rag_evaluations"   # NEW: stores judge scores

# ── LLM (answer layer + judge) ────────────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=2000,
)

print("✓ Config loaded")
print(f"  DB         : {DB_NAME}.{COLLECTION_NAME}")
print(f"  Eval store : {DB_NAME}.{EVAL_COLLECTION}")


## Cell 4 — Ingest PDF with Metadata Tagging
**Improvement #1:** Every chunk is now tagged with `source_file`, `page`, and `topic`.  
This enables filtered retrieval later (e.g. only return chunks about a specific topic).


In [ ]:
def load_and_ingest(pdf_path: str, topic: str = "general"):
    """
    Load a PDF, chunk it, tag each chunk with metadata,
    embed with HuggingFace and store in MongoDB Atlas.
    
    Args:
        pdf_path : path to the PDF file
        topic    : label stored in chunk metadata for filtered retrieval
    Returns:
        MongoDBAtlasVectorSearch instance
    """
    client     = MongoClient(MONGODB_URI)
    collection = client[DB_NAME][COLLECTION_NAME]

    # 1. Load PDF
    loader = PyPDFLoader(pdf_path)
    pages  = loader.load()
    print(f"  Loaded {len(pages)} pages from '{pdf_path}'")

    # 2. Filter very short pages (noise)
    pages = [p for p in pages if len(p.page_content.split()) > 20]
    print(f"  Kept   {len(pages)} pages after cleaning")

    # 3. Chunk
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
    )
    chunks = splitter.split_documents(pages)
    print(f"  Created {len(chunks)} chunks")

    # 4. Improvement #1 — tag every chunk with metadata
    source_name = os.path.basename(pdf_path)
    for chunk in chunks:
        chunk.metadata["source_file"] = source_name
        chunk.metadata["topic"]       = topic
        # page is already set by PyPDFLoader in chunk.metadata["page"]

    # 5. Embed (HuggingFace — free, 768 dims)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )

    # 6. Store in MongoDB Atlas
    vectorstore = MongoDBAtlasVectorSearch.from_documents(
        chunks,
        embeddings,
        collection=collection,
        index_name=INDEX_NAME,
    )
    print(f"✓ Ingested {len(chunks)} chunks into MongoDB")
    print(f"  Metadata tags: source_file='{source_name}', topic='{topic}'")
    return vectorstore


# ── Run ingestion ─────────────────────────────────────────────────────────
# Change the path and topic to match your PDF
PDF_PATH = "./sample_files/mongodb.pdf"
TOPIC    = "mongodb"

# Uncomment to run:
# vectorstore = load_and_ingest(PDF_PATH, topic=TOPIC)
print("⚠ Uncomment the last line to run ingestion")


## Cell 5 — Connect to Existing Vector Store
Run this cell (instead of Cell 4) on subsequent sessions — avoids re-embedding every run.


In [ ]:
def get_vectorstore():
    """Connect to an already-populated MongoDB Atlas vector store."""
    client     = MongoClient(MONGODB_URI)
    collection = client[DB_NAME][COLLECTION_NAME]
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )
    vs = MongoDBAtlasVectorSearch(
        collection=collection,
        embedding=embeddings,
        index_name=INDEX_NAME,
    )
    print("✓ Connected to existing vector store")
    return vs

vectorstore = get_vectorstore()


## Cell 6 — MongoDB Atlas: Vector Search Index
Before querying, make sure the vector search index exists in Atlas.

**Steps:**
1. Atlas dashboard → your cluster → **Search Indexes**
2. Click **Create Search Index** → **JSON Editor**
3. Name: `vector_index`
4. Paste the JSON below and click **Create**
5. Wait ~1 min for status → **Active**

```json
{
  "mappings": {
    "dynamic": true,
    "fields": {
      "embedding": {
        "type": "knnVector",
        "dimensions": 768,
        "similarity": "cosine"
      }
    }
  }
}
```


In [ ]:
print("✓ Reminder: create the vector_index in Atlas if not already done")

## Cell 7 — Retriever Helpers
**Improvement #2:** Two retriever modes — plain similarity and metadata-filtered.


In [ ]:
def get_retriever(vectorstore, k: int = 5, topic_filter: str = None):
    """
    Return a LangChain retriever.
    
    Args:
        vectorstore  : MongoDBAtlasVectorSearch instance
        k            : number of chunks to retrieve
        topic_filter : if set, only return chunks whose metadata.topic matches
    """
    search_kwargs = {"k": k}

    # Improvement #2 — filtered retrieval using metadata
    if topic_filter:
        search_kwargs["pre_filter"] = {"topic": {"$eq": topic_filter}}
        print(f"  Retriever: top-{k}, filtered to topic='{topic_filter}'")
    else:
        print(f"  Retriever: top-{k}, no filter")

    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs=search_kwargs,
    )

# Default retriever (used across the notebook)
retriever = get_retriever(vectorstore, k=5, topic_filter=TOPIC)
print("✓ Retriever ready")


## Cell 8 — RAG Function with LLM Answer Layer
**Improvement #3:** Instead of returning raw chunks, the retrieved context is passed to an LLM  
which synthesises a proper answer — exactly like the DualLens `RAG()` pattern.


In [ ]:
RAG_SYSTEM_PROMPT = """You are a helpful assistant specialising in technical documentation.
Use ONLY the provided context to answer the user's question.
If the answer is not in the context, say: "I don't have enough information to answer that."
Be specific, concise, and reference relevant details from the documents.
"""

def RAG(question: str, retriever=retriever, return_context: bool = False):
    """
    Full RAG pipeline: retrieve → synthesise with LLM → return answer.
    
    Args:
        question       : user's question
        retriever      : LangChain retriever object
        return_context : if True also return the raw context string (needed for evaluation)
    Returns:
        answer string, or (answer, context) tuple when return_context=True
    """
    # 1. Retrieve relevant chunks
    chunks  = retriever.invoke(question)
    context = "\n\n".join([c.page_content for c in chunks])

    if not chunks:
        answer = "No relevant documents found. Check your vector index and ingestion."
        return (answer, context) if return_context else answer

    # 2. Build prompt
    user_prompt = f"""###Context
{context}

###Question
{question}
"""
    messages = [
        ("system", RAG_SYSTEM_PROMPT),
        ("human",  user_prompt),
    ]

    # 3. LLM answer synthesis (Improvement #3)
    try:
        response = llm.invoke(messages)
        answer   = response.content
    except Exception as e:
        answer = f"LLM error: {e}"

    return (answer, context) if return_context else answer


# ── Quick test ────────────────────────────────────────────────────────────
test_q = "What is MongoDB and what is it used for?"
print(f"Question: {test_q}\n")
print(RAG(test_q))


## Cell 9 — LLM-as-Judge: Groundedness & Relevance
**Improvement #4 & #5:** A second LLM call evaluates every answer on two axes:
- **Groundedness (1–5):** Is every claim supported by the retrieved context?
- **Relevance (1–5):** Does the answer directly address the question?

This reusable `evaluate_rag()` function replaces the duplicated code pattern in DualLens.


In [ ]:
JUDGE_PROMPTS = {
    "groundedness": """You are an evaluation assistant assessing groundedness.
Groundedness: every claim in the answer must be directly traceable to the context.

Rate 1-5:
1 - Completely ungrounded
2 - Mostly ungrounded  
3 - Partially grounded
4 - Mostly grounded
5 - Fully grounded

Respond ONLY with valid JSON: {"score": <1-5>, "reasoning": "<brief explanation>"}""",

    "relevance": """You are an evaluation assistant assessing relevance.
Relevance: the answer must directly address the question asked.

Rate 1-5:
1 - Completely irrelevant
2 - Mostly irrelevant
3 - Partially relevant
4 - Mostly relevant
5 - Fully relevant

Respond ONLY with valid JSON: {"score": <1-5>, "reasoning": "<brief explanation>"}""",
}

def evaluate_rag(question: str, answer: str, context: str, criterion: str = "groundedness") -> dict:
    """
    LLM-as-Judge evaluation for a single RAG response.
    
    Args:
        question  : the original user question
        answer    : the RAG-generated answer
        context   : the retrieved context string
        criterion : 'groundedness' or 'relevance'
    Returns:
        dict with keys: score (int 1-5), reasoning (str)
    """
    if criterion not in JUDGE_PROMPTS:
        raise ValueError(f"criterion must be 'groundedness' or 'relevance', got '{criterion}'")

    user_msg = f"""###Question
{question}

###Context
{context}

###Answer
{answer}
"""
    messages = [
        ("system", JUDGE_PROMPTS[criterion]),
        ("human",  user_msg),
    ]

    try:
        response = llm.invoke(messages)
        # Strip markdown fences if present
        raw = response.content.strip().strip("```json").strip("```").strip()
        result = json.loads(raw)
        result["criterion"] = criterion
        return result
    except json.JSONDecodeError:
        return {"criterion": criterion, "score": 0, "reasoning": f"Parse error: {response.content}"}
    except Exception as e:
        return {"criterion": criterion, "score": 0, "reasoning": str(e)}


def evaluate_full(question: str, answer: str, context: str) -> dict:
    """Run both groundedness and relevance evaluation and return combined result."""
    g = evaluate_rag(question, answer, context, "groundedness")
    r = evaluate_rag(question, answer, context, "relevance")
    return {
        "question":              question,
        "answer":                answer,
        "groundedness_score":    g["score"],
        "groundedness_reason":   g["reasoning"],
        "relevance_score":       r["score"],
        "relevance_reason":      r["reasoning"],
        "average_score":         round((g["score"] + r["score"]) / 2, 2),
    }


# ── Test the judge ────────────────────────────────────────────────────────
test_question = "What is a MongoDB collection?"
answer, context = RAG(test_question, return_context=True)

print(f"Question : {test_question}")
print(f"Answer   : {answer[:300]}...\n")

eval_result = evaluate_full(test_question, answer, context)
print(f"Groundedness : {eval_result['groundedness_score']}/5 — {eval_result['groundedness_reason']}")
print(f"Relevance    : {eval_result['relevance_score']}/5  — {eval_result['relevance_reason']}")
print(f"Average      : {eval_result['average_score']}/5")


## Cell 10 — Store Evaluation Scores in MongoDB
**Improvement:** Scores are persisted to a separate `rag_evaluations` collection.  
This builds an evaluation log you can query and plot over time.


In [ ]:
from datetime import datetime, timezone

def store_evaluation(eval_result: dict):
    """Persist an evaluation result to MongoDB."""
    client     = MongoClient(MONGODB_URI)
    collection = client[DB_NAME][EVAL_COLLECTION]
    doc = {**eval_result, "timestamp": datetime.now(timezone.utc)}
    collection.insert_one(doc)
    print(f"✓ Stored evaluation (groundedness={eval_result['groundedness_score']}, "
          f"relevance={eval_result['relevance_score']})")


def load_evaluations() -> pd.DataFrame:
    """Load all stored evaluation results as a DataFrame."""
    client     = MongoClient(MONGODB_URI)
    collection = client[DB_NAME][EVAL_COLLECTION]
    docs = list(collection.find({}, {"_id": 0}))
    if not docs:
        print("No evaluations stored yet.")
        return pd.DataFrame()
    df = pd.DataFrame(docs)
    return df


# ── Run full eval pipeline and store ─────────────────────────────────────
eval_questions = [
    "What is the difference between a database and a collection in MongoDB?",
    "How do you create an index in MongoDB?",
    "What is a replica set?",
    "How does MongoDB store data?",
    "What is aggregation in MongoDB?",
]

all_evals = []
for q in eval_questions:
    print(f"\nEvaluating: {q[:60]}...")
    ans, ctx = RAG(q, return_context=True)
    result   = evaluate_full(q, ans, ctx)
    store_evaluation(result)
    all_evals.append(result)

print("\n✓ All evaluations complete and stored")


## Cell 11 — Evaluation Score Dashboard
**Improvement #6:** Visual summary of groundedness and relevance scores across all test questions —  
similar to the DualLens financial metrics bar charts.


In [ ]:
def plot_evaluation_dashboard(evals: list[dict]):
    """Plot groundedness, relevance, and average scores for all evaluated questions."""
    if not evals:
        print("No evaluations to plot.")
        return

    df = pd.DataFrame(evals)
    labels = [f"Q{i+1}" for i in range(len(df))]
    x      = range(len(df))
    width  = 0.25

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("RAG Evaluation Dashboard — LLM-as-Judge Scores", fontsize=14, fontweight="bold")

    # ── Left: grouped bar chart ───────────────────────────────────────────
    ax = axes[0]
    b1 = ax.bar([i - width for i in x], df["groundedness_score"], width, label="Groundedness", color="#3B8BD4", alpha=0.85)
    b2 = ax.bar([i         for i in x], df["relevance_score"],    width, label="Relevance",    color="#1D9E75", alpha=0.85)
    b3 = ax.bar([i + width for i in x], df["average_score"],      width, label="Average",      color="#EF9F27", alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 5.5)
    ax.axhline(y=4, color="gray", linestyle="--", linewidth=0.8, label="Good threshold (4)")
    ax.set_ylabel("Score (1–5)")
    ax.set_xlabel("Question")
    ax.set_title("Scores per Question")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # ── Right: averages summary ────────────────────────────────────────────
    ax2 = axes[1]
    means  = [df["groundedness_score"].mean(), df["relevance_score"].mean(), df["average_score"].mean()]
    colors = ["#3B8BD4", "#1D9E75", "#EF9F27"]
    bars   = ax2.bar(["Groundedness", "Relevance", "Average"], means, color=colors, alpha=0.85)
    ax2.set_ylim(0, 5.5)
    ax2.axhline(y=4, color="gray", linestyle="--", linewidth=0.8, label="Good threshold (4)")
    ax2.set_ylabel("Mean Score (1–5)")
    ax2.set_title("Overall Mean Scores")
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, means):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 f"{val:.2f}", ha="center", va="bottom", fontweight="bold")

    plt.tight_layout()
    plt.savefig("rag_evaluation_dashboard.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Dashboard saved as rag_evaluation_dashboard.png")


plot_evaluation_dashboard(all_evals)

# ── Print question legend ─────────────────────────────────────────────────
print("\nQuestion legend:")
for i, e in enumerate(all_evals):
    g = e['groundedness_score']
    r = e['relevance_score']
    print(f"  Q{i+1} [{g}/5 G | {r}/5 R] {e['question'][:70]}")


## Cell 12 — Load & Plot Historical Evaluations from MongoDB
Pull all stored evaluations from the `rag_evaluations` collection and re-plot.


In [ ]:
history_df = load_evaluations()

if not history_df.empty:
    print(f"Loaded {len(history_df)} evaluations from MongoDB\n")
    print(history_df[["question", "groundedness_score", "relevance_score", "average_score"]].to_string(index=False))

    # Plot history
    plot_evaluation_dashboard(history_df.to_dict("records"))
else:
    print("Run Cell 10 first to generate and store evaluations.")


## Cell 13 — Interactive Chat Interface
Full chat loop using the improved RAG pipeline with LLM-synthesised answers.  
Type `eval` after any answer to run the judge on the last response.  
Type `quit` to exit.


In [ ]:
def chat(topic_filter: str = TOPIC):
    """
    Interactive RAG chat loop with optional inline evaluation.
    Commands:
        eval  — evaluate the last answer (groundedness + relevance)
        quit  — exit
    """
    local_retriever = get_retriever(vectorstore, k=5, topic_filter=topic_filter)
    last_q, last_a, last_ctx = None, None, None

    print("\n" + "="*60)
    print("  MongoDB RAG Chat  |  Improved Version")
    print("  Commands: 'eval' to judge last answer, 'quit' to exit")
    print("="*60 + "\n")

    while True:
        question = input("You: ").strip()

        if not question:
            continue

        if question.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break

        if question.lower() == "eval":
            if last_q is None:
                print("  No answer to evaluate yet. Ask a question first.\n")
                continue
            print("  Running LLM-as-Judge evaluation...\n")
            result = evaluate_full(last_q, last_a, last_ctx)
            store_evaluation(result)
            print(f"  Groundedness : {result['groundedness_score']}/5 — {result['groundedness_reason']}")
            print(f"  Relevance    : {result['relevance_score']}/5  — {result['relevance_reason']}")
            print(f"  Average      : {result['average_score']}/5\n")
            continue

        # Normal RAG answer
        answer, context = RAG(question, retriever=local_retriever, return_context=True)
        last_q, last_a, last_ctx = question, answer, context

        print(f"\nAssistant: {answer}")
        print("  (type 'eval' to judge this answer)\n")


# Uncomment to start the chat:
# chat()
print("⚠ Uncomment chat() above to start the interactive session")


## Summary of Improvements

| # | Improvement | Where |
|---|-------------|-------|
| 1 | Metadata tagging (source, page, topic) on every chunk | `load_and_ingest()` |
| 2 | Filtered retrieval via MongoDB `pre_filter` | `get_retriever()` |
| 3 | LLM answer synthesis — no more raw chunk dumps | `RAG()` |
| 4 | Groundedness judge (LLM-as-Judge, 1–5) | `evaluate_rag()` |
| 5 | Relevance judge (LLM-as-Judge, 1–5) | `evaluate_rag()` |
| 6 | Scores stored in MongoDB `rag_evaluations` | `store_evaluation()` |
| 7 | Evaluation dashboard with matplotlib | `plot_evaluation_dashboard()` |
| 8 | `eval` command inside chat loop for instant feedback | `chat()` |

### Architecture at a glance
```
PDF ──► chunk ──► metadata tag ──► HuggingFace embed ──► MongoDB Atlas
                                                              │
Question ──► filtered retriever ──► top-k chunks ──► LLM synthesis ──► Answer
                                                              │
                                                        Judge LLM
                                                    groundedness + relevance
                                                              │
                                                    MongoDB rag_evaluations
                                                              │
                                                    matplotlib dashboard
```
